In [4]:
import os
import re
import glob
import pandas as pd

# Folder where your CSVs are (current directory is usually fine in notebooks)
BASE_DIR = "."

# Grab all "* responses.csv" files
paths = sorted(glob.glob(os.path.join(BASE_DIR, "* responses.csv")))

if not paths:
    raise SystemExit("No '* responses.csv' files found in BASE_DIR")

def llm_name_from_filename(path: str) -> str:
    # "gpt52 responses.csv" -> "gpt52"
    base = os.path.basename(path)
    name = re.sub(r"\s*responses\.csv$", "", base, flags=re.IGNORECASE)
    return name.strip()

frames = []
for p in paths:
    llm = llm_name_from_filename(p)
    df = pd.read_csv(p)

    if "number" not in df.columns or "choice" not in df.columns:
        raise ValueError(f"{p} must contain columns 'number' and 'choice'")

    df = df[["number", "choice"]].copy()
    df["number"] = pd.to_numeric(df["number"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["number"])
    df = df.drop_duplicates(subset=["number"], keep="last")
    df = df.rename(columns={"choice": llm})

    frames.append(df)

# Outer-join all on number, so you keep every question number seen anywhere
merged = frames[0]
for df in frames[1:]:
    merged = merged.merge(df, on="number", how="outer")

# Put number first, then model columns alphabetically
model_cols = sorted([c for c in merged.columns if c != "number"])
merged = merged[["number"] + model_cols].sort_values("number")

out_path = os.path.join(BASE_DIR, "all_llm_choices.csv")
merged.to_csv(out_path, index=False)

print("Wrote:", out_path)
merged.head()


Wrote: .\all_llm_choices.csv


,number,claude35sonnet,claudesonnet45,deepseekv32,gpt41,gpt52,gpt5pro,llama323binstruct,qwen3maxthinking
0,1,D,D,D,D,D,D,D,D
1,2,D,D,D,D,D,D,D,D
2,3,D,D,D,D,D,D,D,D
3,4,B,B,B,B,B,B,B,B
4,5,D,D,D,D,D,D,D,D


In [5]:
import pandas as pd

# Load the merged file you created earlier
df = pd.read_csv("all_llm_choices.csv")

# All model columns (everything except "number")
model_cols = [c for c in df.columns if c != "number"]

# Count A–E per model
counts = (
    df[model_cols]
    .apply(lambda s: s.astype(str).str.upper().str.strip().value_counts())
    .reindex(["A", "B", "C", "D", "E"])
    .fillna(0)
    .astype(int)
)

# Add totals + percentages (optional but handy)
counts.loc["TOTAL"] = counts.sum(axis=0)

# Save
counts.to_csv("option_counts_by_model.csv")

print("Wrote: option_counts_by_model.csv")
counts


Wrote: option_counts_by_model.csv


,claude35sonnet,claudesonnet45,deepseekv32,gpt41,gpt52,gpt5pro,llama323binstruct,qwen3maxthinking
A,4,2,2,2,0,0,1,1
B,15,23,19,26,11,15,26,20
C,2,2,6,10,3,3,12,7
D,129,121,121,111,135,131,111,121
E,0,2,2,1,1,1,0,1
TOTAL,150,150,150,150,150,150,150,150


In [1]:
import pandas as pd

# Non-informed merged choices file: one column per model, one row per question
INPUT = "all_llm_choices.csv"
OUT = "non_informed_counts_by_hierarchy.csv"

df = pd.read_csv(INPUT)

# Models = all columns except question number
model_cols = [c for c in df.columns if c != "number"]

# Correct hierarchy split based on Nscenarios.csv / main.csv:
# Q1–Q75 = higher-power roles
# Q76–Q150 = lower-power roles
df["hierarchy"] = df["number"].apply(
    lambda n: "Higher (1–75)" if int(n) <= 75 else "Lower (76–150)"
)

# Clean choices
for c in model_cols:
    df[c] = df[c].astype(str).str.upper().str.strip()
    df.loc[~df[c].isin(["A", "B", "C", "D", "E"]), c] = pd.NA

OPTIONS = ["A", "B", "C", "D", "E"]

def counts_table(sub_df: pd.DataFrame) -> pd.DataFrame:
    ct = sub_df[model_cols].apply(lambda s: s.value_counts(dropna=True))
    ct = ct.reindex(OPTIONS).fillna(0).astype(int)
    return ct

counts_higher = counts_table(df[df["hierarchy"] == "Higher (1–75)"])
counts_lower = counts_table(df[df["hierarchy"] == "Lower (76–150)"])

# Main table: rows are (group, option), columns are models
combined = pd.concat(
    {
        "Higher (1–75)": counts_higher,
        "Lower (76–150)": counts_lower,
    },
    axis=0,
)

combined.index.names = ["Group", "Option"]

# Optional difference block: Higher - Lower
# Negative values mean the option was selected more often in lower-power cases.
diff = (counts_higher - counts_lower).reindex(OPTIONS)
combined = pd.concat(
    [combined, pd.concat({"Δ (Higher−Lower)": diff}, axis=0)]
)

combined.index.names = ["Group", "Option"]

combined.to_csv(OUT)
print("Wrote:", OUT)
print(combined)
combined


Wrote: non_informed_counts_by_hierarchy.csv
                         claude35sonnet  claudesonnet45  deepseekv32  gpt41  \
Group            Option                                                       
Higher (1–75)    A                    1               1            0      0   
                 B                    3               8            7     13   
                 C                    2               2            3      6   
                 D                   69              64           65     56   
                 E                    0               0            0      0   
Lower (76–150)   A                    3               1            2      2   
                 B                   12              15           12     13   
                 C                    0               0            3      4   
                 D                   60              57           56     55   
                 E                    0               2            2      1   
Δ (Highe

claude35sonnet  claudesonnet45  deepseekv32  gpt41  \
Group            Option                                                       
Higher (1–75)    A                    1               1            0      0   
                 B                    3               8            7     13   
                 C                    2               2            3      6   
                 D                   69              64           65     56   
                 E                    0               0            0      0   
Lower (76–150)   A                    3               1            2      2   
                 B                   12              15           12     13   
                 C                    0               0            3      4   
                 D                   60              57           56     55   
                 E                    0               2            2      1   
Δ (Higher−Lower) A                   -2               0           -2     -2   
                 B                   -9              -7           -5      0   
                 C                    2               2            0      2   
                 D                    9               7            9      1   
                 E                    0              -2           -2     -1   

                         gpt52  gpt5pro  llama323binstruct  qwen3maxthinking  
Group            Option                                                       
Higher (1–75)    A           0        0                  1                 0  
                 B           5        7                 13                 7  
                 C           2        2                  4                 4  
                 D          68       66                 57                64  
                 E           0        0                  0                 0  
Lower (76–150)   A           0        0                  0                 1  
                 B           6        8                 13                13  
                 C           1        1                  8                 3  
                 D          67       65                 54                57  
                 E           1        1                  0                 1  
Δ (Higher−Lower) A           0        0                  1                -1  
                 B          -1       -1                  0                -6  
                 C           1        1                 -4                 1  
                 D           1        1                  3                 7  
                 E          -1       -1                  0                -1